# Module 3 · Chapter 1 — 분포의 사다리: 베르누이→이항→포아송→정규

본문(.md)과 짝이 되는 실습 노트북입니다.
- **본문**: 네 분포의 직관·정의·수식과 상호 연결 관계
- **이 노트북**: 신약 임상시험 시나리오로 각 분포를 직접 시뮬레이션

## 0. 환경 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

rng = np.random.default_rng(seed=42)
sns.set_theme(context="notebook", style="whitegrid")

## 1. 베르누이 분포 — 환자 한 명의 부작용 발생

부작용 발생 확률 p=0.15. 한 명의 환자 = 베르누이(0.15).

In [ ]:
p = 0.15  # 부작용 발생 확률

# PMF: P(X=0), P(X=1)
bern = stats.bernoulli(p)
print("=== 베르누이(p=0.15) ===")
print(f"P(부작용 없음) = P(X=0) = {bern.pmf(0):.2f}")
print(f"P(부작용 있음) = P(X=1) = {bern.pmf(1):.2f}")
print(f"기댓값 E[X] = {bern.mean():.2f},  분산 Var[X] = {bern.var():.4f}")

# 1000번 시뮬레이션
samples = bern.rvs(size=1000, random_state=42)
print(f"\n시뮬레이션 1000회 → 부작용 발생: {samples.sum()}회 ({samples.mean():.3f})")

## 2. 이항 분포 — 20명 임상에서 몇 명에게 부작용이?

In [ ]:
n_patients = 20
binom_dist = stats.binom(n=n_patients, p=p)

k_vals = np.arange(0, n_patients + 1)
pmf_vals = binom_dist.pmf(k_vals)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(k_vals, pmf_vals, color="steelblue", alpha=0.8, edgecolor="white")
axes[0].axvline(binom_dist.mean(), color="tomato", linestyle="--",
                linewidth=2, label=f"기댓값 = {binom_dist.mean():.1f}")
axes[0].set_title(f"이항 분포 Binomial(n={n_patients}, p={p})")
axes[0].set_xlabel("부작용 발생 환자 수")
axes[0].set_ylabel("확률")
axes[0].legend()

# 직접 계산 vs scipy 비교
from math import comb
def binom_pmf_manual(k, n, p):
    return comb(n, k) * (p**k) * ((1-p)**(n-k))

manual = [binom_pmf_manual(k, n_patients, p) for k in range(5)]
scipy_ = [binom_dist.pmf(k) for k in range(5)]

cmp_df = pd.DataFrame({"k": range(5), "직접 계산": manual, "scipy": scipy_})
print("=== 이항 PMF 비교 (k=0..4) ===")
print(cmp_df.to_string(index=False, float_format="{:.6f}".format))

axes[1].plot(range(5), manual, "o-", label="직접 계산", markersize=8)
axes[1].plot(range(5), scipy_, "x--", label="scipy", markersize=8, linewidth=2)
axes[1].set_title("직접 계산 vs scipy (k=0..4)")
axes[1].set_xlabel("k")
axes[1].set_ylabel("P(X=k)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. 포아송 분포 — 하루 응급실 방문 건수

λ = n×p = 1000명 × 0.003 = 3명/일 (희귀한 중증 부작용)

In [ ]:
lam = 3  # 하루 평균 응급 방문 3명
poisson_dist = stats.poisson(mu=lam)

k_range = np.arange(0, 15)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(k_range, poisson_dist.pmf(k_range), color="mediumseagreen", alpha=0.8)
ax.axvline(lam, color="tomato", linestyle="--", linewidth=2,
           label=f"λ(평균=분산) = {lam}")
ax.set_title(f"포아송 분포 Poisson(λ={lam})")
ax.set_xlabel("하루 응급실 방문 수")
ax.set_ylabel("확률")
ax.legend()
plt.tight_layout()
plt.show()

print(f"기댓값 = {poisson_dist.mean()}, 분산 = {poisson_dist.var()}")
print("→ 포아송의 특징: 평균 = 분산 = λ")
print(f"P(하루 0명 방문) = {poisson_dist.pmf(0):.4f}")
print(f"P(하루 5명 이상) = {1 - poisson_dist.cdf(4):.4f}")

## 4. CLT — 이항 분포가 정규에 수렴하는 과정

n이 커질수록 Binomial(n, p)의 모양이 정규 분포를 닮아갑니다.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
ns = [5, 20, 50, 200]
colors = ["steelblue", "tomato", "mediumseagreen", "mediumpurple"]

for ax, n_val, color in zip(axes, ns, colors):
    binom_n = stats.binom(n=n_val, p=p)
    mu = binom_n.mean()
    sigma = binom_n.std()

    k_vals = np.arange(0, n_val + 1)
    ax.bar(k_vals, binom_n.pmf(k_vals), color=color, alpha=0.6, label=f"Binom(n={n_val})")

    # 정규 근사 오버레이
    x_norm = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
    ax.plot(x_norm, stats.norm.pdf(x_norm, mu, sigma),
            color="black", linewidth=2, label="정규 근사")

    ax.set_title(f"n = {n_val}")
    ax.set_xlabel("k")
    if ax == axes[0]:
        ax.set_ylabel("P(X=k)")
    ax.legend(fontsize=8)

plt.suptitle(f"이항(n, p={p})에서 n이 커질수록 정규에 수렴", fontsize=13)
plt.tight_layout()
plt.show()

## 5. 직접 답해 보기

1. 이항 분포를 포아송으로 근사하는 것이 적절한 조건은 무엇인가?
2. 포아송 분포에서 "평균 = 분산"이 성립하지 않는 실제 데이터를 본다면 어떤 분포가 더 적합할까?
3. CLT가 성립하려면 어떤 조건이 필요한가? (독립성, n의 크기 등)